# XGBoost v2.4 — Mixed Training (CIC + Generated CSVs)

## Why v2.4?
The v2.3 model had 34.9% attack recall on the generated test CSVs because:
- It was trained **only on CIC-DDoS2019/CIC-IoT2023** patterns
- The generated CSVs have **different feature distributions** (mild/synthetic attack values)

**v2.4 Solution**: Include the generated attack and benign CSVs in the training data.
This teaches the model to recognise BOTH real CIC patterns AND the generated patterns.

## What Changed v2.3 → v2.4
| Change | Detail |
|---|---|
| **Training data source** | CIC balanced dataset + 3 generated attack CSVs + 4 generated benign CSVs |
| `scale_pos_weight` | `1.5` → **`2.0`** |
| `BETA` | `3.0` → **`3.0`** (same) |
| `max_depth` | `6` → **`7`** (more complex patterns) |

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import glob
import gc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, fbeta_score
)
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
plt.style.use('ggplot')
print('[OK] Libraries loaded')

## 1. Load & Merge All Training Data
**Sources:**
1. CIC Balanced dataset (main corpus)
2. Generated attack CSVs (add real generated patterns)
3. Generated benign CSVs (prevent false positives on generated benign)

In [ ]:
# ── 1A. Main CIC balanced dataset ────────────────────────────────────────────
CIC_PATH = r"C:\Users\shenal\Downloads\reseraach\PCAPS_Used\Final_Balanced_Attack_and_Benign\Final_balanced_Attack_and_Benign_new_Shuffled.csv"

# ── 1B. Generated test CSVs ───────────────────────────────────────────────────
# Attack CSVs — will be labelled ATTACK
GENERATED_ATTACK_PATHS = [
    r"C:\Users\shenal\Downloads\reseraach\Test\dns_attacks_generated_1.csv",
    r"C:\Users\shenal\Downloads\reseraach\Test\dns_attacks_generated_2.csv",
    r"C:\Users\shenal\Downloads\reseraach\Test\dns_attacks_generated_3.csv",
]

# Benign CSVs — will be labelled BENIGN
GENERATED_BENIGN_PATHS = [
    r"C:\Users\shenal\Downloads\reseraach\Test\dns_benign_generated_1.csv",
    r"C:\Users\shenal\Downloads\reseraach\Test\dns_benign_generated_2.csv",
    r"C:\Users\shenal\Downloads\reseraach\Test\dns_benign_generated_3.csv",
    r"C:\Users\shenal\Downloads\reseraach\Test\dns_benign_generated_4.csv",
]

dfs = []

# Load CIC dataset
print(f'[INFO] Loading CIC dataset...')
df_cic = pd.read_csv(CIC_PATH)
print(f'  CIC: {len(df_cic):,} rows')
dfs.append(df_cic)
del df_cic; gc.collect()

# Load and label generated attack CSVs
for path in GENERATED_ATTACK_PATHS:
    try:
        df_a = pd.read_csv(path)
        df_a['label'] = 'ATTACK'   # Force label
        print(f'  Attack: {len(df_a):,} rows — {path.split(chr(92))[-1]}')
        dfs.append(df_a)
        del df_a
    except FileNotFoundError:
        print(f'  [SKIP] Not found: {path}')

# Load and label generated benign CSVs
for path in GENERATED_BENIGN_PATHS:
    try:
        df_b = pd.read_csv(path)
        df_b['label'] = 'BENIGN'   # Force label
        print(f'  Benign: {len(df_b):,} rows — {path.split(chr(92))[-1]}')
        dfs.append(df_b)
        del df_b
    except FileNotFoundError:
        print(f'  [SKIP] Not found: {path}')

# Merge all
df = pd.concat(dfs, ignore_index=True)
del dfs; gc.collect()

print(f'\n[MERGED] Total: {len(df):,} rows')
print(df['label'].value_counts())

## 2. Preprocessing + Selective Log-Transform

In [ ]:
COLS_TO_DROP = ['src_ip', 'dst_ip', 'src_port', 'dst_port', 'protocol_number']
df = df.drop(columns=COLS_TO_DROP, errors='ignore')
df.replace([np.inf, -np.inf], 0, inplace=True)
df.fillna(0, inplace=True)

LABEL_MAP = {'BENIGN': 0, 'ATTACK': 1}
df['label'] = df['label'].str.upper().map(LABEL_MAP).fillna(0).astype(int)
print(f'Labels after encode: {dict(df["label"].value_counts())}')

PROTOCOL_CLASSES = ['DOH', 'DOT', 'TRADITIONAL', 'UNKNOWN', 'TCP', 'UDP']
le_proto = LabelEncoder()
le_proto.fit(PROTOCOL_CLASSES)
df['protocol'] = df['protocol'].astype(str).apply(
    lambda x: x if x in PROTOCOL_CLASSES else 'UNKNOWN'
)
df['protocol'] = le_proto.transform(df['protocol'])

# Selective log-transform (same as v2.3 — dns_queries_per_second kept RAW)
LOG_FEATURES = [
    'bwd_packets_per_sec',
    'flow_bytes_per_sec',
    'flow_packets_per_sec',
    'fwd_packets_per_sec',
    'total_fwd_packets',
    'total_bwd_packets',
]
for col in LOG_FEATURES:
    if col in df.columns:
        df[col] = np.log1p(df[col].clip(lower=0))

# Downcast to save RAM
df[df.select_dtypes('float64').columns] = df.select_dtypes('float64').astype(np.float32)
df[df.select_dtypes('int64').columns]   = df.select_dtypes('int64').astype(np.int32)
print(f'[OK] Preprocessing done. Memory: {df.memory_usage(deep=True).sum()/1e6:.1f} MB')
gc.collect()

## 3. Train / Test Split — Keep Class Balance

In [ ]:
X = df.drop(columns=['label'])
y = df['label']
del df; gc.collect()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
del X, y; gc.collect()

n_benign = (y_train == 0).sum()
n_attack = (y_train == 1).sum()
print(f'Train: BENIGN={n_benign:,}  ATTACK={n_attack:,}')
print(f'Test:  {X_test.shape[0]:,} rows')

## 4. XGBoost v2.4 — scale_pos_weight=2.0

In [ ]:
SCALE_POS_WEIGHT = 2.0   # Attack is 2x more important than benign

model = xgb.XGBClassifier(
    objective          = 'binary:logistic',
    eval_metric        = ['logloss', 'auc'],
    use_label_encoder  = False,

    scale_pos_weight   = SCALE_POS_WEIGHT,

    n_estimators       = 1000,
    learning_rate      = 0.03,
    max_depth          = 7,
    min_child_weight   = 3,
    gamma              = 0.05,

    subsample          = 0.8,
    colsample_bytree   = 0.7,
    colsample_bylevel  = 0.7,
    reg_alpha          = 0.05,
    reg_lambda         = 1.0,

    tree_method        = 'hist',
    random_state       = 42,
)

print(f'[TRAIN] scale_pos_weight={SCALE_POS_WEIGHT}, max_depth=7, n_estimators=1000')
print(f'[INFO] Training on merged CIC + generated CSV data...')
model.fit(
    X_train, y_train,
    eval_set = [(X_test, y_test)],
    verbose  = 100,
)
print('[DONE] Training complete.')
gc.collect()

## 5. Threshold Tuning (beta=3.0)

In [ ]:
y_prob = model.predict_proba(X_test)[:, 1]
BETA = 3.0

thresholds = np.arange(0.05, 0.95, 0.01)
best_thresh, best_score = 0.5, 0
for t in thresholds:
    score = fbeta_score(y_test, (y_prob >= t).astype(int), beta=BETA, zero_division=0)
    if score > best_score:
        best_score, best_thresh = score, t

print(f'[TUNING] Optimal threshold = {best_thresh:.2f}  (F-{BETA} = {best_score:.4f})')

scores = [fbeta_score(y_test, (y_prob >= t).astype(int), beta=BETA, zero_division=0)
          for t in thresholds]
plt.figure(figsize=(10, 4))
plt.plot(thresholds, scores, color='steelblue')
plt.axvline(best_thresh, color='red',  linestyle='--', label=f'Best={best_thresh:.2f}')
plt.axvline(0.5,         color='gray', linestyle=':',  label='Default=0.50')
plt.xlabel('Threshold'); plt.ylabel(f'F-beta (beta={BETA})')
plt.title('v2.4 Threshold Tuning'); plt.legend(); plt.show()

## 6. Balance Report

In [ ]:
y_pred = (y_prob >= best_thresh).astype(int)

print(f'=== CLASSIFICATION REPORT (threshold={best_thresh:.2f}) ===')
print(classification_report(y_test, y_pred,
      target_names=['BENIGN','ATTACK'], digits=4))

auc = roc_auc_score(y_test, y_prob)
print(f'ROC-AUC: {auc:.4f}')

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['BENIGN','ATTACK'], yticklabels=['BENIGN','ATTACK'])
plt.title(f'v2.4 Confusion Matrix @ threshold={best_thresh:.2f}')
plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.tight_layout(); plt.show()

tn, fp, fn, tp = cm.ravel()
benign_recall  = tn / (tn + fp) if (tn + fp) > 0 else 0
attack_recall  = tp / (tp + fn) if (tp + fn) > 0 else 0
false_pos_rate = fp / (fp + tn) if (fp + tn) > 0 else 0

print(f'\n===============================')
print(f' v2.4 BALANCE REPORT')
print(f'===============================')
print(f' BENIGN Recall  : {benign_recall*100:6.1f}%  (target: >90%)')
print(f' ATTACK Recall  : {attack_recall*100:6.1f}%  (target: >90%)')
print(f' False Pos Rate : {false_pos_rate*100:6.1f}%  (target: <10%)')
print(f' ROC-AUC        : {auc:.4f}')
print(f'===============================')

if attack_recall >= 0.90 and benign_recall >= 0.90:
    print('\n[EXCELLENT] Both recalls above 90%!')
elif attack_recall < 0.85:
    print('\n[ADVICE] Raise scale_pos_weight to 3.0 and retrain')
elif benign_recall < 0.85:
    print('\n[ADVICE] Lower scale_pos_weight to 1.5 and retrain')

## 7. Feature Importance

In [ ]:
importance = pd.Series(model.feature_importances_, index=X_train.columns)
top20 = importance.nlargest(20)

plt.figure(figsize=(10, 7))
top20.sort_values().plot(kind='barh', color='steelblue')
plt.title('Top 20 Feature Importances — XGBoost v2.4')
plt.xlabel('Importance'); plt.tight_layout(); plt.show()

print('\n--- Top 10 ---')
for feat, imp in top20.head(10).items():
    print(f'  {feat:<35} {imp:.4f}')

## 8. Save Model Bundle

In [ ]:
bundle = {
    'model':             model,
    'threshold':         best_thresh,
    'log_features':      LOG_FEATURES,
    'protocol_classes':  PROTOCOL_CLASSES,
    'scale_pos_weight':  SCALE_POS_WEIGHT,
    'beta':              BETA,
    'version':           'v2.4',
    'notes':             'Trained on CIC + generated CSVs; scale_pos_weight=2.0',
}

with open('xgb_model_v2.4.pkl', 'wb') as f:
    pickle.dump(bundle, f)

print(f'[SAVED] xgb_model_v2.4.pkl')
print(f'  threshold       = {best_thresh:.2f}')
print(f'  ATTACK Recall   = {attack_recall*100:.1f}%')
print(f'  BENIGN Recall   = {benign_recall*100:.1f}%')
print(f'  ROC-AUC         = {auc:.4f}')
print()
print('Update MODEL_PATH in test_xgb_model_v2.1.py to: xgb_model_v2.4.pkl')